# Lesson 2.6 — 加载 dataset 并构建 batch

本 notebook 覆盖从「dataset 已存在于磁盘上」到「模型可以消费它」之间的这一步：

`LeRobotDataset` → `dataset[index]` → `DataLoader` → batch tensors。

这一步刻意保持机械。这里不涉及任何学习；重点在于确切了解进入 policy 的内容的
shape、dtype 与含义。


## 2.6.1 — 读取 dataset 与单个 frame

`dataset[index]` 返回单个 timestep：`observation.state [42]` 与
`action [8]`，两者都是 `float32`，外加 index 与 timestamp 的记账字段。

这些断言固定了上述 shape，使 schema 变化会直接报错，而不是在错误的内容上
静默训练。


In [1]:
from pathlib import Path
import os

# Notebooks may be started with either the repository root or notebooks/ as the working
# directory. Walk up until the project root is found, so dataset and cache paths never
# resolve to notebooks/datasets or notebooks/.cache by accident.
_cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in (_cwd, *_cwd.parents) if (p / ".git").exists()),
    _cwd.parent if _cwd.name == "notebooks" else _cwd,
)

# Set the Hugging Face cache BEFORE importing lerobot: the `datasets` package snapshots
# its cache location at import time, so configuring it later has no effect and reads fall
# back to the (non-writable) home directory.
HF_CACHE = PROJECT_ROOT / ".cache" / "hf"
(HF_CACHE / "datasets").mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HF_HOME", str(HF_CACHE))
os.environ.setdefault("HF_DATASETS_CACHE", str(HF_CACHE / "datasets"))

print("project root:", PROJECT_ROOT)
print("HF cache    :", HF_CACHE)


project root: /home/bowenyuan/Projects/embodied-ai-learning
HF cache    : /home/bowenyuan/Projects/embodied-ai-learning/.cache/hf


In [2]:
from pathlib import Path

print("Current working directory:", Path.cwd())

Current working directory: /home/bowenyuan/Projects/embodied-ai-learning


In [3]:
from pathlib import Path
from lerobot.datasets.lerobot_dataset import LeRobotDataset


DATASET_ROOT = PROJECT_ROOT / "datasets" / "lerobot" / "pickcube"

print("Project root:", PROJECT_ROOT)
print("Dataset root:", DATASET_ROOT)
print("Dataset exists:", DATASET_ROOT.exists())
print("Data exists:", (DATASET_ROOT / "data").exists())
print("Meta exists:", (DATASET_ROOT / "meta").exists())

dataset = LeRobotDataset(
    repo_id="pickcube",
    root=DATASET_ROOT,
)

print("Dataset length:", len(dataset))
print("Episodes:", dataset.num_episodes)
print("FPS:", dataset.fps)


Project root: /home/bowenyuan/Projects/embodied-ai-learning
Dataset root: /home/bowenyuan/Projects/embodied-ai-learning/datasets/lerobot/pickcube
Dataset exists: True
Data exists: True
Meta exists: True


Dataset length: 50
Episodes: 1
FPS: 50


In [4]:
sample = dataset[0]

print("Sample type:", type(sample))
print("\nKeys:")

for key, value in sample.items():
    shape = getattr(value, "shape", None)
    dtype = getattr(value, "dtype", type(value))
    print(f"{key:30s} shape={str(shape):15s} dtype={dtype}")

Sample type: <class 'dict'>

Keys:
observation.state              shape=torch.Size([42]) dtype=torch.float32
action                         shape=torch.Size([8]) dtype=torch.float32
timestamp                      shape=torch.Size([])  dtype=torch.float32
frame_index                    shape=torch.Size([])  dtype=torch.int64
episode_index                  shape=torch.Size([])  dtype=torch.int64
index                          shape=torch.Size([])  dtype=torch.int64
task_index                     shape=torch.Size([])  dtype=torch.int64
task                           shape=None            dtype=<class 'str'>


In [5]:
state = sample["observation.state"]
action = sample["action"]

print("\nState shape:", state.shape)
print("Action shape:", action.shape)

print("\nState:")
print(state)

print("\nAction:")
print(action)


State shape: torch.Size([42])
Action shape: torch.Size([8])

State:
tensor([ 3.5281e-02,  4.0070e-01,  1.9575e-02, -1.9187e+00,  3.7351e-02,
         2.3366e+00,  8.0440e-01,  4.0000e-02,  4.0000e-02,  0.0000e+00,
         0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
         0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  1.2254e-02,
         3.8011e-02,  1.8215e-01, -1.7689e-02,  9.9980e-01,  4.2880e-03,
         7.9842e-03,  2.6816e-02, -1.9813e-03,  2.8893e-01, -7.4868e-04,
         5.3644e-02,  2.0000e-02,  5.6876e-01,  0.0000e+00,  0.0000e+00,
         8.2250e-01, -1.3002e-02,  1.5633e-02, -1.6215e-01,  2.7564e-02,
        -5.5626e-02,  2.6893e-01])

Action:
tensor([ 0.1791,  0.9838,  0.1854, -0.8309, -0.2816, -0.4202, -0.8190,  0.0052])


In [6]:
assert state.shape == (42,)
assert action.shape == (8,)
assert len(dataset) == 50
assert dataset.num_episodes == 1
assert dataset.fps == 50

print("Single-frame inspection: PASS")

Single-frame inspection: PASS


## 2.6.2 — 将样本组成 batch

`DataLoader` 将样本堆叠成一个 batch，因此 policy 收到
`states [8,42]` 与 `target_actions [8,8]`。新增的前导维度就是 batch
维度；单个样本的语义不变。

这里的 loader 叫 `inspection_loader`，并且显式使用 `shuffle=False`：它只用于
可重复的检查，保持 trajectory 的时间顺序，所以打印出的 `index` /
`episode_index` / `frame_index` / `timestamp` 会按原顺序连续出现，多次运行结果
一致。是否 shuffle 是**训练时**的选择，不是数据格式的属性：`2.7` 会单独构建
自己的 `train_loader`，并在那里使用 `shuffle=True`。

需要记住的边界：一旦引入时间窗口 `[B,T,D]` 或多条 episode，shuffle 就不能再
在 frame 级别做，否则会把相邻 frame 拆到不同 batch，必须改为按 episode（或按
窗口）打乱。

batch 中的 `index` / `episode_index` / `frame_index` 字段让某一行可以
追溯回它的 episode 与时间，这在存在不止一条 trajectory 时立刻变得重要。

In [7]:
from torch.utils.data import DataLoader

BATCH_SIZE = 8

# Inspection loader: shuffle=False keeps trajectory order, so frame_index and
# timestamp stay consecutive and repeated runs are reproducible.
# Shuffling is a training-time decision; 2.7 builds its own train_loader.
inspection_loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)


In [8]:
batch = next(iter(inspection_loader))

print("Batch type:", type(batch))
print("\nBatch fields:")

for key, value in batch.items():
    if hasattr(value, "shape"):
        print(
            f"{key:30s}",
            f"shape={str(value.shape):15s}",
            f"dtype={value.dtype}",
        )
    else:
        print(
            f"{key:30s}",
            f"type={type(value)}",
            f"value={value}",
        )

Batch type: <class 'dict'>

Batch fields:
observation.state              shape=torch.Size([8, 42]) dtype=torch.float32
action                         shape=torch.Size([8, 8]) dtype=torch.float32
timestamp                      shape=torch.Size([8]) dtype=torch.float32
frame_index                    shape=torch.Size([8]) dtype=torch.int64
episode_index                  shape=torch.Size([8]) dtype=torch.int64
index                          shape=torch.Size([8]) dtype=torch.int64
task_index                     shape=torch.Size([8]) dtype=torch.int64
task                           type=<class 'list'> value=['pick up the cube', 'pick up the cube', 'pick up the cube', 'pick up the cube', 'pick up the cube', 'pick up the cube', 'pick up the cube', 'pick up the cube']


In [9]:
states = batch["observation.state"]
actions = batch["action"]

print("States:", states.shape)
print("Actions:", actions.shape)

assert states.shape == (8, 42)
assert actions.shape == (8, 8)

print("Batch inspection: PASS")

States: torch.Size([8, 42])
Actions: torch.Size([8, 8])
Batch inspection: PASS


In [10]:
print("Global indices:", batch["index"])
print("Episode indices:", batch["episode_index"])
print("Frame indices:", batch["frame_index"])
print("Timestamps:", batch["timestamp"])
print("Tasks:", batch["task"])

Global indices: tensor([0, 1, 2, 3, 4, 5, 6, 7])
Episode indices: tensor([0, 0, 0, 0, 0, 0, 0, 0])
Frame indices: tensor([0, 1, 2, 3, 4, 5, 6, 7])
Timestamps: tensor([0.0000, 0.0200, 0.0400, 0.0600, 0.0800, 0.1000, 0.1200, 0.1400])
Tasks: ['pick up the cube', 'pick up the cube', 'pick up the cube', 'pick up the cube', 'pick up the cube', 'pick up the cube', 'pick up the cube', 'pick up the cube']
